# Interactive 3D

In [[B10CBE41-6C45-4FFA-9D4A-1026925C0130|Lights, Camera, Animation]] your scenes moved on their own. Now they will start listening to *you*. By the end of this notebook you will have built a small game with a player you steer, coins to collect, a score on screen, and sound effects.

All of it rests on one idea, the **event**. An event is something that happens while the scene is running: a shape is clicked, a key is pressed, two shapes touch. You write a function that says what to do, and hand it to the scene. That function is called a **handler**, and the scene calls it for you whenever the event happens.

Run the cell below and click the ball. Press **Stop** (■) when you are done.

In [ ]:
from codetto import graphics3d

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
scene.set_ground(length=10, width=10)

ball = graphics3d.Shapes.Sphere(diameter=2, segments=24)
ball.set_position(0, 1, 0)
ball.set_color('#4488ff')

def change_color():
  ball.set_color('#e94560')

ball.on_click(change_color)
scene.add(ball)

scene.run()

# Clicks

Look at the line that connects the click to the function:

```python
ball.on_click(change_color)
```

There are no brackets after `change_color`. Writing `change_color()` would run the function immediately. Writing just its name hands the function over, so the scene can run it later, when the click happens.

Two rules to remember:

- Call `on_click` **before** `scene.add(shape)`.
- Handlers only work while the scene is running, so the cell must end with `scene.run()`.

A handler can do anything a normal function can: count, make decisions, change several shapes. The balloon below counts its clicks. Each click pumps it up a little, and the sixth click pops it with `balloon.remove()`, which takes a shape out of the scene for good.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#87ceeb')
scene.set_ground(length=10, width=10)

balloon = graphics3d.Shapes.Sphere(diameter=1, segments=24)
balloon.set_position(0, 2.5, 0)
balloon.set_color('#e94560')

pumps = 0

def pump():
  global pumps
  pumps += 1
  if pumps < 6:
    size = 1 + pumps * 0.3
    balloon.set_scale(size, size, size)
  else:
    balloon.remove()

balloon.on_click(pump)
scene.add(balloon)

scene.run()

# Keys

Key presses belong to the whole scene, not to one shape, so you register them on the scene:

```python
scene.on_key(graphics3d.Key.LEFT, go_left)
scene.on_key('w', go_forward)
```

Special keys have names in `graphics3d.Key`:

| Name | Key |
|---|---|
| `Key.LEFT`, `Key.RIGHT`, `Key.UP`, `Key.DOWN` | The arrow keys |
| `Key.SPACE` | Space bar |
| `Key.ENTER` | Enter |
| `Key.ESCAPE` | Esc |

Letters and digits are plain strings, such as `'w'` or `'1'`.

In the cell below the arrow keys move a ball one step at a time. Each handler reads where the ball is with `get_position`, then puts it one step along. Space sends it back to the start.

> **Tip:** If the keys do nothing, click once on the scene so it is listening to the keyboard. Once a scene uses `on_key`, the arrow keys no longer turn the camera, but dragging with the mouse still does.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#0f3460')
ground = scene.set_ground(length=16, width=16)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(8)

player = graphics3d.Shapes.Sphere(diameter=0.8, segments=24)
player.set_position(0, 0.4, 0)
player.set_color('#e94560')
scene.add(player)

def move(dx, dz):
  x, y, z = player.get_position()
  player.set_position(x + dx, y, z + dz)

def go_left():
  move(-1, 0)

def go_right():
  move(1, 0)

def go_forward():
  move(0, 1)

def go_back():
  move(0, -1)

def go_home():
  player.set_position(0, 0.4, 0)

scene.on_key(graphics3d.Key.LEFT, go_left)
scene.on_key(graphics3d.Key.RIGHT, go_right)
scene.on_key(graphics3d.Key.UP, go_forward)
scene.on_key(graphics3d.Key.DOWN, go_back)
scene.on_key(graphics3d.Key.SPACE, go_home)

scene.run()

# Writing on the Screen

Games show a score, a timer, or a message on top of the action. That layer is called a **HUD** (heads-up display). Every scene has one: a see-through sheet in front of the 3D view that you can draw on.

```python
hud = scene.get_context('2d')

hud.clear()                          # wipe the sheet
hud.fill_style = '#ffffff'           # choose a color
hud.font = '24px sans-serif'         # choose a text size
hud.fill_text('Score: 0', 20, 40)    # write at x=20, y=40
```

The HUD is flat, so it uses two numbers measured in pixels: `x` from the left edge and `y` down from the top. `hud.fill_rect(x, y, width, height)` draws a filled rectangle, which makes a good backing panel for text.

Anything you draw stays until you wipe it. To update a score, call `hud.clear()` and draw everything again. The usual way is to put all the drawing in one function and call it whenever something changes. Redraw only when something has changed, not on every frame, or the text may flicker.

The HUD has the same drawing commands as a 2D canvas. [[538D0FC0-A15B-4B4D-926C-1FBE4BF2BC73|Drawing on a Canvas]] covers all of them.

This is the scene from before, with a HUD that shows the ball's position and counts your moves.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#0f3460')
ground = scene.set_ground(length=16, width=16)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(8)

player = graphics3d.Shapes.Sphere(diameter=0.8, segments=24)
player.set_position(0, 0.4, 0)
player.set_color('#e94560')
scene.add(player)

hud = scene.get_context('2d')
moves = 0

def draw_hud():
  x, y, z = player.get_position()
  hud.clear()
  hud.fill_style = 'rgba(0, 0, 0, 0.5)'
  hud.fill_rect(10, 10, 230, 70)
  hud.fill_style = '#ffffff'
  hud.font = '20px sans-serif'
  hud.fill_text(f'Position: x={x:.0f}  z={z:.0f}', 22, 38)
  hud.fill_text(f'Moves: {moves}', 22, 66)

def move(dx, dz):
  global moves
  x, y, z = player.get_position()
  player.set_position(x + dx, y, z + dz)
  moves += 1
  draw_hud()

def go_left():
  move(-1, 0)

def go_right():
  move(1, 0)

def go_forward():
  move(0, 1)

def go_back():
  move(0, -1)

scene.on_key(graphics3d.Key.LEFT, go_left)
scene.on_key(graphics3d.Key.RIGHT, go_right)
scene.on_key(graphics3d.Key.UP, go_forward)
scene.on_key(graphics3d.Key.DOWN, go_back)

draw_hud()
scene.run()

# Collisions

A **collision** event happens at the moment two shapes begin to touch.

```python
ball.on_collide(wall, bounce)    # call bounce() when the ball touches the wall
```

Three rules:

- Both shapes must already be in the scene, so call `on_collide` **after** `scene.add` for both of them.
- Call it **before** `scene.run()`.
- The handler is called once each time the shapes *start* touching. It is not called again while they stay in contact. If they move apart and meet again, it is called again.

Shapes do not stop each other on their own. Without a handler, the ball below would glide straight through the walls. The `bounce` handler is what turns it around, by flipping `direction` between `1` and `-1`.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#0f3460')
scene.set_ground(length=8, width=16)

ball = graphics3d.Shapes.Sphere(diameter=1, segments=24)
ball.set_position(0, 0.5, 0)
ball.set_color('#f5a623')
scene.add(ball)

left_wall = graphics3d.Shapes.Box(width=1, height=2, depth=4)
left_wall.set_position(-6, 1, 0)
left_wall.set_material(graphics3d.Material.Bricks.DarkClay)
scene.add(left_wall)

right_wall = graphics3d.Shapes.Box(width=1, height=2, depth=4)
right_wall.set_position(6, 1, 0)
right_wall.set_material(graphics3d.Material.Bricks.DarkClay)
scene.add(right_wall)

hud = scene.get_context('2d')
direction = 1
bounces = 0

def draw_hud():
  hud.clear()
  hud.fill_style = '#ffffff'
  hud.font = '24px sans-serif'
  hud.fill_text(f'Bounces: {bounces}', 20, 40)

def bounce():
  global direction, bounces
  direction = -direction
  bounces += 1
  draw_hud()

ball.on_collide(left_wall, bounce)
ball.on_collide(right_wall, bounce)

@scene.on_frame
def animate(dt):
  x, y, z = ball.get_position()
  ball.set_position(x + direction * 4 * dt, y, z)

draw_hud()
scene.run()

# Sound Effects

A click that makes a noise feels far more real than a silent one. Codetto's `audio` library plays sounds, and `samples` has a collection of ready-made ones:

```python
from codetto import audio, samples

audio.play(samples.Sounds.Laser)
```

Call `audio.play` inside a handler and the sound plays when the event happens.

Each of the three shapes below needs a handler that plays a *different* sound. Rather than write three nearly identical functions, the cell uses `partial` from Python's `functools` library. `partial(play, samples.Sounds.Laser)` means "the `play` function, with its sound already filled in", which is exactly the kind of no-brackets function that `on_click` expects.

> **Good to know:** `audio.play` waits for the sound to finish, and the scene pauses while it waits. Keep sound effects short. There is much more about sound in [[807A26DA-D106-4E6B-9921-8B6BCC8054E2|Playing Notes and Sounds]].

In [ ]:
from functools import partial
from codetto import audio, samples

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
scene.set_ground(length=8, width=12)

def play(sound):
  audio.play(sound)

laser = graphics3d.Shapes.Cone(diameter=1.4, height=2)
laser.set_position(-3.5, 1, 0)
laser.set_color('#e94560')
laser.on_click(partial(play, samples.Sounds.Laser))
scene.add(laser)

coin = graphics3d.Shapes.Sphere(diameter=1.8, segments=24)
coin.set_position(0, 0.9, 0)
coin.set_color('#f5a623')
coin.on_click(partial(play, samples.Sounds.CoinPickup))
scene.add(coin)

bell = graphics3d.Shapes.Box(width=1.6, height=1.6, depth=1.6)
bell.set_position(3.5, 0.8, 0)
bell.set_color('#44cc88')
bell.on_click(partial(play, samples.Sounds.Clang))
scene.add(bell)

scene.run()

# Putting It Together: Coin Dash

Here is a complete game built from the pieces in this notebook.

- **Keys** move the player around the field, and the camera follows.
- **Collisions** between the player and each coin call `collect`, which removes the coin, adds to the score, and plays a sound. The `coins` list keeps track of the coins that are still on the field, so a coin can only be collected once. `partial(collect, coin)` gives each coin its own handler, the same trick as the sound board.
- **Every frame**, a timer counts up until the last coin is gone. The HUD is only redrawn when the number of whole seconds changes.
- The **HUD** shows the score and the time.

Run the cell, then use the **arrow keys** to move the red ball. Collect all six coins as fast as you can.

Then make the game your own:

- Add more coins to the `coin_spots` list.
- Add a second way to steer with the `'w'`, `'a'`, `'s'`, and `'d'` keys.
- Add a hazard: a red box that sends the player back to the start when touched.
- Show a message when the time goes over 20 seconds.
- Change the look of the field with materials, a sky, and lighting.

For the last step, let gravity loose on your scenes in [[CABF4887-1FD4-4BDB-8E11-0B34B237F890|3D Physics]].

In [ ]:
from functools import partial
from codetto import audio, samples

scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.PURE_SKY)
ground = scene.set_ground(length=16, width=16)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(8)

player = graphics3d.Shapes.Sphere(diameter=0.8, segments=24)
player.set_position(0, 0.4, 0)
player.set_color('#e94560')
scene.add(player)
scene.camera.follow(player, distance=14)

coin_spots = [(3, 0), (5, 4), (-2, 5), (-6, 1), (-4, -5), (4, -6)]
coins = []
hud = scene.get_context('2d')
score = 0
seconds = 0

def draw_hud():
  hud.clear()
  hud.fill_style = 'rgba(0, 0, 0, 0.5)'
  hud.fill_rect(10, 10, 220, 70)
  hud.fill_style = '#ffffff'
  hud.font = '20px sans-serif'
  hud.fill_text(f'Coins: {score} / {len(coin_spots)}', 22, 38)
  hud.fill_text(f'Time: {int(seconds)}', 22, 66)
  if score == len(coin_spots):
    hud.fill_style = '#f5a623'
    hud.font = '32px sans-serif'
    hud.fill_text('You got them all!', 22, 120)

def collect(coin):
  global score
  if coin not in coins:
    return  # this coin was already collected
  coins.remove(coin)
  coin.remove()
  score += 1
  draw_hud()
  if score == len(coin_spots):
    audio.play(samples.Sounds.Correct)
  else:
    audio.play(samples.Sounds.CoinPickup)

for x, z in coin_spots:
  coin = graphics3d.Shapes.Cylinder(diameter=0.7, height=0.15)
  coin.set_position(x, 0.4, z)
  coin.set_rotation(x=90)
  coin.set_color('#f5a623')
  scene.add(coin)
  coins.append(coin)
  player.on_collide(coin, partial(collect, coin))

def move(dx, dz):
  x, y, z = player.get_position()
  x = max(-7, min(7, x + dx))
  z = max(-7, min(7, z + dz))
  player.set_position(x, y, z)

scene.on_key(graphics3d.Key.LEFT, partial(move, -1, 0))
scene.on_key(graphics3d.Key.RIGHT, partial(move, 1, 0))
scene.on_key(graphics3d.Key.UP, partial(move, 0, 1))
scene.on_key(graphics3d.Key.DOWN, partial(move, 0, -1))

@scene.on_frame
def tick(dt):
  global seconds
  if score < len(coin_spots):
    before = int(seconds)
    seconds += dt
    if int(seconds) != before:
      draw_hud()

draw_hud()
scene.run()

# Check Your Understanding